In [2]:
from pulp import *
import random

class Pattern:
    def __init__(self, name, lengths=None, lenOpts=None, totalRollLength=20):
        self.name = name.replace(" ", "_")
        self.lenOpts = lenOpts or ["5", "7", "9"]
        self.totalRollLength = totalRollLength
        self.lengthsdict = dict(zip(self.lenOpts, lengths))

    def __str__(self):
        return f"Pattern(name={self.name}, lengths={self.lengthsdict})"

    def trim(self):
        return self.totalRollLength - sum(
            [int(i) * int(self.lengthsdict[i]) for i in self.lengthsdict]
        )

def generateExample(num_lengths=3, max_demand=100, totalRollLength=50):
    """
    Generates input data for the Cutting Stock problem.
    """
    # Generate possible roll lengths
    lengths = []
    while len(lengths) < num_lengths:
        length = str(random.randint(3, 10))
        if length not in lengths:
            lengths.append(length)
    # Generate roll demand
    rollData = {length: (random.randint(1, max_demand), random.uniform(0.1, 1.0)) for length in lengths}
    
    # Generate initial patterns
    Patterns = []
    for i, length in enumerate(lengths):
        pattern_lengths = [0] * len(lengths)
        pattern_lengths[i] = totalRollLength // int(length) # how many times can you fit there
        Patterns.append(Pattern(f"P{i+1}", pattern_lengths, lenOpts=lengths, totalRollLength=totalRollLength))

    return rollData, Patterns, lengths, totalRollLength

rollData, Patterns, lenOpts, totalRollLength  = generateExample()
print("demand for each roll (len : (demand, weigth)): ",rollData)
print("lengths: ",lenOpts)
print('total roll length: ',totalRollLength)
print("pattern: ",str(Patterns[0]))

demand for each roll (len : (demand, weigth)):  {'4': (24, 0.9158787921237963), '9': (20, 0.362729641946301), '6': (79, 0.37723596207933974)}
lengths:  ['4', '9', '6']
total roll length:  50
pattern:  Pattern(name=P1, lengths={'4': 12, '9': 0, '6': 0})


In [57]:

def masterDualSolve(Patterns, rollData, lenOpts, totalRollLength, relax=True, iteration=0):
    """
    Solves the dual formulation of the master problem.
    """
    global constraint_counter
    rollDemand = {key: val[0] for key, val in rollData.items()}

    # Create the dual optimization problem
    prob = LpProblem("Cutting_Stock_Dual", LpMaximize)
    
    # Decision variables - one for each length (dual variables)
    dualVars = LpVariable.dicts("DualVar", lenOpts, 0, None, LpContinuous)

    # Objective: maximize sum of (demand * dual variable) for each length
    prob += lpSum([rollDemand[j] * dualVars[j] for j in lenOpts])

    # Constraints: for each pattern, sum of (pattern coefficients * dual variables) <= 1
    for pattern in Patterns:
        constraint_name = f"Pattern_{pattern.name}_Iter{iteration}_{constraint_counter}"
        constraint_counter += 1
        prob += (
            lpSum([pattern.lengthsdict[j] * dualVars[j] for j in lenOpts]) <= 1,
            constraint_name
        )

    prob.solve()

    # Return the dual values
    duals = {j: value(dualVars[j]) for j in lenOpts}
    return duals


def masterSolve(Patterns, rollData, lenOpts, totalRollLength, relax=True, iteration=0):
    """
    Solves the master problem of the Column Generation method.
    """
    global constraint_counter
    rollDemand = {key: val[0] for key, val in rollData.items()}

    # Create the optimization problem
    prob = LpProblem("Cutting_Stock_Problem", LpMinimize)
    vartype = LpContinuous if relax else LpInteger
    pattVars = LpVariable.dicts("Pattern", Patterns, 0, None, vartype)

    # Objective: minimize the number of large rolls used
    prob += lpSum([pattVars[i] * 1 for i in Patterns])

    # Constraints: satisfy demand for each roll length
    for j in lenOpts:
        constraint_name = f"Min{j}_Iter{iteration}_{constraint_counter}"  # Ensure absolute uniqueness
        constraint_counter += 1
        prob += (
            lpSum([pattVars[i] * i.lengthsdict[j] for i in Patterns])
            >= rollDemand[j],
            constraint_name,
        )

    prob.solve()

    if relax:
        duals = {
            j: prob.constraints[f"Min{j}_Iter{iteration}_{constraint_counter - len(lenOpts) + idx}"].pi
            for idx, j in enumerate(lenOpts)
        }
        return duals
    else:
        varsdict = {v.name: v.varValue for v in prob.variables()}
        return value(prob.objective), varsdict

def subSolve(Patterns, duals, lenOpts, totalRollLength):
    """
    Solves the subproblem of the Column Generation method.
    """
    # Create the optimization problem
    prob = LpProblem("SubProb", LpMinimize)
    _vars = LpVariable.dicts("Roll Length", lenOpts, 0, None, LpInteger)
    trim = LpVariable("Trim", 0, None, LpInteger)

    # Objective: minimize the cost of a new pattern
    prob += (1 - lpSum([_vars[i] * duals[i] for i in lenOpts])), "Objective"

    # Constraint: ensure the total length of the new roll matches the allowed length
    prob += (
        lpSum([_vars[i] * int(i) for i in lenOpts]) + trim
        == totalRollLength,
        "lengthEquate",
    )

    # Save the coefficients, matrix, and RHS (right-hand side) in LP history
    matrix = [[int(i) for i in lenOpts]]
    rhs = [totalRollLength]
    coefficients = {i: duals[i] for i in lenOpts}
    lp_history.append({"coefficients": coefficients, "matrix": matrix, "rhs": rhs})

    prob.solve()

    # Generate a new pattern based on the solution
    newPattern = {i: int(_vars[i].varValue) for i in lenOpts}
    if value(prob.objective) < -(10**-5):
        morePatterns = True
        if newPattern not in [p.lengthsdict for p in Patterns]:
            Patterns.append(Pattern(f"P{len(Patterns) + 1}", [newPattern[i] for i in lenOpts], lenOpts, totalRollLength))
    else:
        morePatterns = False

    return Patterns, morePatterns


def columnGeneration(Patterns, rollData, lenOpts, totalRollLength):
    """
    Executes the iterative Column Generation process.
    """
    iterations = 0
    while True:
        # Solve the master problem
        duals = masterSolve(Patterns, rollData, lenOpts, totalRollLength, relax=True)

        # Solve the subproblem
        Patterns, morePatterns = subSolve(Patterns, duals, lenOpts, totalRollLength)

        # If no new patterns are generated, terminate
        if not morePatterns:
            break
        iterations += 1

    return Patterns


# Global variable to store LP history for the subproblem (coefficients, matrix, and right-hand side)
lp_history = []

# Global counter for unique constraint names
constraint_counter = 0

# Generate random input data
#rollData, Patterns, lenOpts, totalRollLength = generateExample()

print("num of initial patterns: ", len(Patterns))
print(Patterns)
# Solve the problem using Column Generation
optimalPatterns = columnGeneration(Patterns, rollData, lenOpts, totalRollLength)
print("num of optimal patterns: ", len(optimalPatterns))
# Display results
print("\nGenerated Roll Data:")
for length, (demand, _) in rollData.items():
    print(f"Length: {length}, Demand: {demand}")

print("\nOptimal Patterns:")
for pattern in optimalPatterns:
    print(pattern)

# Print the LP history for the subproblem (coefficients, matrix, RHS)
print("\nSubproblem LP History:")
for i, lp in enumerate(lp_history, 1):
    print(f"--- Subproblem Iteration {i} ---")
    print(f"Coefficients: {lp['coefficients']}")
    print(f"Matrix: {lp['matrix']}")
    print(f"RHS: {lp['rhs']}")

num of initial patterns:  6
[<__main__.Pattern object at 0x7337b35cb910>, <__main__.Pattern object at 0x7337b35c9ab0>, <__main__.Pattern object at 0x7337b35c85e0>, <__main__.Pattern object at 0x7337b17a89d0>, <__main__.Pattern object at 0x7337b17a8100>, <__main__.Pattern object at 0x7337b17a9fc0>]
Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /home/jan/miniconda3/envs/ipmgnn/lib/python3.10/site-packages/pulp/solverdir/cbc/linux/64/cbc /tmp/4b76d6fbca44473c9156b1127a3b4137-pulp.mps -timeMode elapsed -branch -printingOptions all -solution /tmp/4b76d6fbca44473c9156b1127a3b4137-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 8 COLUMNS
At line 25 RHS
At line 29 BOUNDS
At line 30 ENDATA
Problem MODEL has 3 rows, 6 columns and 10 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Presolve 3 (0) rows, 6 (0) columns and 10 (0) elements
Perturbing problem by 0.001% of 1.5 - lar

In [83]:
def generateRandomGraph(num_vertices, difficulty='medium'):
    """
    Generate a random graph instance with specified size and difficulty level
    
    Parameters:
    num_vertices: int - Number of vertices in the graph
    difficulty: str - Difficulty level ('easy', 'medium', 'hard') affecting edge density
    
    Returns:
    edges: List of tuples representing edges
    num_vertices: int - Number of vertices
    """
    import random
    
    # Set edge probability based on difficulty
    if difficulty.lower() == 'easy':
        edge_prob = 0.2  # Sparse graph
    elif difficulty.lower() == 'hard':
        edge_prob = 0.7  # Dense graph
    else:  # medium
        edge_prob = 0.4  # Moderate density
        
    edges = []
    # Generate random edges
    for i in range(num_vertices):
        for j in range(i+1, num_vertices):
            if random.random() < edge_prob:
                edges.append((i,j))
                
    return edges, num_vertices

def masterSolveDual(colorings, edges, num_vertices):
    """
    Solve the dual of the master problem (Set Covering)
    colorings: List of valid colorings
    edges: List of edges in the graph
    num_vertices: Number of vertices
    Returns: Dual values (shadow prices) for each vertex constraint
    """
    # Create the dual master LP
    master_dual = LpProblem("MasterDual", LpMaximize)
    
    # Decision variables - one for each vertex (dual variables)
    pi = LpVariable.dicts("pi", range(num_vertices))
    
    # Objective: Maximize sum of dual variables
    master_dual += lpSum(pi[v] for v in range(num_vertices))
    
    # Constraints: For each coloring, sum of dual vars <= 1
    for i in range(len(colorings)):
        master_dual += lpSum(pi[v] for v in range(num_vertices) 
                                if colorings[i][v] == 1) <= 1
    
    master_dual.solve(PULP_CBC_CMD(msg=False))
    # Print objective value
    #print(value(master_dual.objective))
    # Get optimal dual values
    duals = [pi[v].value() for v in range(num_vertices)]
    return duals,value(master_dual.objective)



def masterSolve(colorings, edges, num_vertices, relax=True):
    """
    Solve the master problem (Set Covering)
    colorings: List of valid colorings
    edges: List of edges in the graph
    num_vertices: Number of vertices
    relax: If True, solve LP relaxation. If False, solve IP
    Returns: Dual values for each vertex constraint
    """
    # Create the master LP
    master = LpProblem("Master", LpMinimize)
    
    # Decision variables - one for each coloring
    if relax:
        x = LpVariable.dicts("x", range(len(colorings)), lowBound=0)
    else:
        x = LpVariable.dicts("x", range(len(colorings)), lowBound=0, cat='Integer')

    # Objective: Minimize number of colorings used
    master += lpSum(x[i] for i in range(len(colorings)))

    # Constraints: Each vertex must be colored exactly once
    for v in range(num_vertices):
        constraint_name = f"Vertex_{v}"
        master += (lpSum(x[i] for i in range(len(colorings)) 
                           if colorings[i][v] == 1) == 1,
                    constraint_name)
    
    master.solve(PULP_CBC_CMD(msg=False))
    
    if relax:
        # Get dual values for each vertex constraint
        duals = [master.constraints[f'Vertex_{v}'].pi for v in range(num_vertices)]
        return duals
    return None

def subSolve(colorings, duals, edges, num_vertices,tolerance=1e-6):
    """
    Solve the subproblem (Maximum Weight Independent Set)
    Returns: Updated colorings list and boolean indicating if new coloring found
    """
    # Create subproblem
    sub = LpProblem("Sub", LpMaximize)
    
    # Decision variables for each vertex (1 if in independent set, 0 otherwise)
    y = LpVariable.dicts("y", range(num_vertices), cat='Binary')
    
    # Store problem data for history
    coefficients = {str(v): float(duals[v]) for v in range(num_vertices)}
    matrix = []
    rhs = []
    
    # Objective: Maximize sum of dual values
    sub += lpSum(duals[v] * y[v] for v in range(num_vertices))
    
    # Constraints: Adjacent vertices can't both be in independent set
    for i, j in edges:
        sub += y[i] + y[j] <= 1
        matrix.append([i, j])
        rhs.append(1)
    
    sub.solve(PULP_CBC_CMD(msg=False))
    
    # Store the LP data in history
    lp = None
    
    # If reduced cost > 1, we found a new coloring
    objective_value = value(sub.objective)
    relative_gap = abs(objective_value - 1) / max(1, abs(objective_value))
    
    if relative_gap > tolerance:
        try:    
            new_coloring = [int(value(y[v])) for v in range(num_vertices)]
            colorings.append(new_coloring)
            lp={
                'coefficients': coefficients,
                'matrix': matrix,
                'rhs': rhs,
                'sol':new_coloring
            }
            return colorings, True, lp
        except:
            return colorings, False, lp # TODO: fix this
    return colorings, False, lp

def graphColoring(edges, num_vertices,use_dual=False):
    """Main column generation procedure for graph coloring"""
    # Initialize with singleton colorings
    colorings = [[1 if i == j else 0 for j in range(num_vertices)] 
                for i in range(num_vertices)]
    lp_history = []
    
    iterations = 0
    lp = {}
    last_valeu = 10000
    while True:
        if use_dual:
            # Solve master problem to get dual values
            duals,value = masterSolveDual(colorings, edges, num_vertices)
            lp_history.append(lp)

        else:
            # Solve master problem to get dual values
            duals = masterSolve(colorings, edges, num_vertices, relax=True)
        # Solve subproblem to find new coloring
        colorings, new_coloring,lp = subSolve(colorings, duals, edges, num_vertices)
        if not new_coloring:
            break
        iterations += 1
    
    return colorings,lp_history

def final_solve(colorings, num_vertices):
    # Solve final integer program with generated colorings
    master = LpProblem("Graph_Coloring_IP", LpMinimize)

    # Decision variables - whether to use each coloring
    x = [LpVariable(f"x_{i}", 0, 1, LpBinary) for i in range(len(colorings))]

    # Objective - minimize number of colorings used
    master += lpSum(x)

    # Each vertex must be colored exactly once
    for v in range(num_vertices):
        master += lpSum(coloring[v] * x[i] for i, coloring in enumerate(colorings)) == 1

    # Solve integer program
    master.solve(PULP_CBC_CMD(msg=False))
    return master, x

# Generate and solve example
use_dual = False
edges, num_vertices = generateRandomGraph(30, 'medium')
use_dual = True

print("\nGraph:")
print(f"Vertices: {num_vertices}")
print(f"Edges: {edges}")

# Solve using column generation
colorings,lp_history = graphColoring(edges, num_vertices,use_dual=use_dual)



master, x = final_solve(colorings, num_vertices)
print("\nInteger solution:")
print(f"Chromatic number: {value(master.objective)}")
print("Used colorings:")
for i in range(len(colorings)):
    if value(x[i]) > 0.5:  # Account for numerical precision
        print(f"Coloring {i}: {colorings[i]}")




Graph:
Vertices: 30
Edges: [(0, 1), (0, 2), (0, 4), (0, 5), (0, 10), (0, 12), (0, 14), (0, 15), (0, 17), (0, 22), (0, 29), (1, 5), (1, 7), (1, 9), (1, 14), (1, 16), (1, 17), (1, 18), (1, 21), (1, 22), (2, 3), (2, 4), (2, 5), (2, 6), (2, 8), (2, 9), (2, 10), (2, 11), (2, 14), (2, 15), (2, 16), (2, 17), (2, 20), (2, 21), (2, 22), (2, 25), (2, 28), (3, 6), (3, 10), (3, 11), (3, 14), (3, 15), (3, 17), (3, 18), (3, 19), (3, 20), (3, 27), (3, 28), (3, 29), (4, 7), (4, 14), (4, 15), (4, 18), (4, 21), (4, 23), (4, 25), (4, 27), (5, 6), (5, 9), (5, 10), (5, 13), (5, 14), (5, 15), (5, 16), (5, 17), (5, 18), (5, 20), (5, 21), (5, 23), (5, 24), (5, 25), (5, 26), (5, 27), (5, 28), (6, 10), (6, 12), (6, 14), (6, 17), (6, 18), (6, 20), (6, 26), (7, 8), (7, 9), (7, 12), (7, 15), (7, 16), (7, 17), (7, 20), (7, 22), (7, 24), (7, 25), (7, 27), (8, 9), (8, 12), (8, 13), (8, 15), (8, 16), (8, 22), (8, 23), (8, 24), (8, 25), (8, 27), (9, 11), (9, 18), (9, 20), (9, 23), (9, 26), (9, 27), (9, 29), (10, 14), 

In [80]:
import random

problems = []
for _ in range(5000):
    # Random size between 10 and 30 vertices
    size = random.randint(10, 30)
    # Generate random graph
    edges, num_vertices = generateRandomGraph(size, 'medium')
    problems.append((edges, num_vertices))
    # Solve using column generation
# Save results to pickle file
import pickle
with open('graph_coloring_problems.pkl', 'wb') as f:
    pickle.dump(problems, f)


In [81]:
with open('graph_coloring_problems.pkl', 'rb') as f:
    problems = pickle.load(f)

In [82]:
from tqdm import tqdm
lp_history_total = []
for problem in tqdm(problems):
    edges, num_vertices = problem
    colorings,lp_history = graphColoring(edges, num_vertices,use_dual=use_dual)
    lp_history_total.append(lp_history)

  1%|          | 45/5000 [01:29<2:44:42,  1.99s/it]


KeyboardInterrupt: 